In [ ]:
import torch 
import numpy as np 
import matplotlib.pyplot as plt
from itertools import islice
import json
from tqdm.auto import tqdm

from cryo_sbi.wpa_simulator.cryo_em_simulator import cryo_em_simulator
from cryo_sbi import CryoEmSimulator
from cryo_sbi.inference.models import build_models
import cryo_sbi.utils.estimator_utils as est_utils
from cryo_sbi.inference.priors import get_image_priors, PriorLoader
from cryo_sbi.inference.models.build_models import build_nle_flow_model

### Testing the NLE flow

Here we test the quality of the NLE flow. First we need to load the trained model.

In [ ]:
# Parameters
device='cuda:0'
# Load trained model
estimator = est_utils.load_estimator(
    "training_parameters_nle.json",
    build_models.build_nle_flow_model,
    "tutorial_estimator.pt",
    device=device,
)

### Image generation for testing

Now we reload the orginal models in `hsp90_models.pt`, center them, and create a new set of models containing the close and open states in the desired population. We start with a 75%-25% mixture of close and open states 

In [ ]:
def center_models(models):
    """
    Remove center of mass from each model.
    
    Args:
        models: torch.Tensor of shape [num_models, 3, N]
                where 3 = (x, y, z) and N = number of atoms
    
    Returns:
        centered_models: torch.Tensor of same shape, centered at origin
    """
    # Compute center of mass for each model
    # Mean over atoms (dim=2) -> [num_models, 3]
    com = models.mean(dim=2, keepdim=True)  # [num_models, 3, 1]
    
    # Subtract center of mass
    centered_models = models - com
    
    return centered_models

In [ ]:
def create_weighted_ensemble(models, w, total_models, verbose=True):
    """
    Create an ensemble of models by repeating them according to weights.
    
    Parameters:
        models: torch.Tensor of shape (n_models, ...)
        w: np.array or list of weights (length n_models)
        total_models: int, desired total number of models in ensemble

        verbose: bool, whether to print summary
    
    Returns:
        models_ensemble: torch.Tensor of repeated models, shape (total_models, ...)
        w_actual: np.array of actual weights after integer rounding
    """

    # Convert w to numpy array and normalize
    w = np.array(w)
    w = w / w.sum()
    
    # Get non-zero indices
    indices = np.where(w > 0)[0]
    active_weights = w[indices]
    
    # Calculate counts (rounded to integers)
    counts = np.round(active_weights * total_models).astype(int)
    
    # Handle edge case: if rounding gives 0 total models
    if counts.sum() == 0:
        counts[np.argmax(active_weights)] = 1  # Give at least one to the largest weight
    
    # Recalculate actual weights from integer counts
    w_actual = np.zeros_like(w)
    w_actual[indices] = counts / counts.sum()
    
    # Create models tensor with repetitions
    model_list = [models[idx].unsqueeze(0).repeat(count, 1, 1) 
                  for idx, count in zip(indices, counts) if count > 0]
    
    # Concatenate to create ensemble
    models_ensemble = torch.cat(model_list, dim=0)
    
    # Print summary
    if verbose:
        print(f"\nInput weights (w):")
        print(f"  Non-zero indices: {indices.tolist()}")
        print(f"  Non-zero weights: {active_weights.tolist()}")
        print(f"\nCounts for {total_models} total models:")
        print(f"  Counts: {counts.tolist()}")
        print(f"  Actual total: {counts.sum()}")
        print(f"  Model shape: {models_ensemble.shape}")
        print(f"\nActual weights (w'):")
        print(f"  Full w': {w_actual.tolist()}")
    
    return models_ensemble, w_actual

In [ ]:
# load models from file
models = torch.load("../hsp90_models.pt").to(device)
# center models (just to be sure)
models = center_models(models)
# Define population weights
w = np.array([0.75, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.25])

# Create weighted ensemble
models_ensemble, w_actual = create_weighted_ensemble(
    models=models,
    w=w,
    total_models=4,
    verbose=True
)

# save weigthed ensemble - needed for image generation
torch.save(models_ensemble, "models.pt")

We will now simulate the cryo-EM images with our generated models. The simulation is done by the class `CryoEmSimulator` and the simulation is run by the `simulate` function. The class `CryoEmSimulator` takes as input a config file with the simulation parameters. The config file used here is `simulation_parameters.json`.

The following parameters are used in the simulation:
```
{
    "N_PIXELS": 128,
    "PIXEL_SIZE": 1.5,
    "SIGMA": [1.0, 1.0],
    "MODEL_FILE": "models.pt",
    "SHIFT": 0.0,
    "DEFOCUS": [0.5, 2.0],
    "SNR": [0.3, 0.3],
    "AMP": 0.1,
    "B_FACTOR": [1.0, 1.0]
}
```

In [ ]:
# create simulator
simulator = CryoEmSimulator(
    "simulation_parameters.json"
)  # creating simulator with simulation parameters
# generate 100k images
images, parameters = simulator.simulate(
    num_sim=100000, return_parameters=True
)  # simulating images and save parameters
# be sure images are on device
images = images.to(device)

#### Visualize the simulated images

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(10, 10), sharex=True, sharey=True)
for idx, ax in enumerate(axes.flatten()):
    ax.imshow(images[idx], vmin=-3, vmax=3, cmap="gray")
    ax.axis("off")

### Evaluate likelihood matrix

In [ ]:
def evaluate_likelihood_pairwise(
    estimator: torch.nn.Module,
    images: torch.Tensor,        # shape: [N_images, H, W]
    models: torch.Tensor,        # shape: [N_models, 3, N_atoms] - same as training
    batch_size_images: int = 64,
    device: str = "cpu"
) -> torch.Tensor:
    """
    Evaluate log p(X_i | theta_j) for all pairs (i, j).
    Models are the same as used in training, so indices = 0, 1, 2, ..., N_models-1
    
    Args:
        estimator: trained NLE model
        images: tensor of images
        models: tensor of models (same as training, same order)
        batch_size_images: batch size for images
        batch_size_models: batch size for models
        device: computation device
    
    Returns:
        log_probs: shape [N_images, N_models] 
                   log_probs[i, j] = log p(image_i | model_j)
    """
    estimator.eval()
    estimator.to(device)
    
    N_images = len(images)
    N_models = len(models)
    
    # Model indices are simply 0, 1, 2, ..., N_models-1
    model_indices = torch.arange(N_models, dtype=torch.long).to(device)
    
    # Initialize results matrix
    log_probs = torch.zeros(N_images, N_models).to(device)
    
    with torch.no_grad():
        # Iterate over image batches
        for i in tqdm(range(0, N_images, batch_size_images), desc="Images"):
            i_end = min(i + batch_size_images, N_images)
            batch_images = images[i:i_end]  # [B_img, H, W]
            
            # Iterate over all models
            for j in range(0, N_models):            
                # Evaluate likelihood
                log_probs[i:i_end, j] = estimator(batch_images, model_indices[j:j+1])
    
    return log_probs

Before evaluating the likelihood, we reload all the 20 models. No need to center them, only the indexes matter, as the flow has learned the relationship between a cryo-EM image and the model index (from 0 to 19).

In [ ]:
# reload models from file
models = torch.load("../hsp90_models.pt").to(device)

In [ ]:
# evaluate likelihood matrix
log_probs_matrix = evaluate_likelihood_pairwise(
    estimator,
    images,
    models,
    batch_size_images=100,
    device=device
)
# transpose to [N_models : N_images]
log_probs_matrix = log_probs_matrix.T

### Population inference (BioEM style)

In [ ]:
class WeightOptimizer:
    """
    Numerically stable optimizer for:
    L = -sum_i log(sum_j w_j * p_ij) + theta * sum_j w_j * log(w_j / w_j^0)
    """
    
    def __init__(self, log_p, w0=None, theta=0.0, device='cpu'):
        """
        Args:
            log_p: tensor of shape (n_j, n_i) containing log(p_ij)
            w0: prior weights of shape (n_j,), normalized. If None, uses uniform weights.
            theta: regularization parameter (float), default 0.0
            device: 'cpu' or 'cuda'
        """
        self.device = device
        self.log_p = torch.tensor(log_p, dtype=torch.float64, device=device)
        self.n_j, self.n_i = self.log_p.shape
        
        # Initialize w0 as uniform if not provided - used only with theta>0
        if w0 is None:
            self.w0 = torch.ones(self.n_j, dtype=torch.float64, device=device) / self.n_j
            if(theta>0): print(f"Prior weights w0 not specified, using uniform: w0 = 1/{self.n_j}")
        else:
            self.w0 = torch.tensor(w0, dtype=torch.float64, device=device)
        
        self.theta = torch.tensor(theta, dtype=torch.float64, device=device)

    def compute_loss(self, w):
        """
        Compute loss given weights w (must be normalized, sum to 1, non-negative)
        Loss is normalized: average log-likelihood per sample and average KL per weight
        """
        eps = 1e-15
        
        # First term: -(1/n_i) * sum_i log(sum_j w_j * p_ij)
        log_w = torch.log(w + eps)  # shape (n_j,)
        log_terms = log_w.unsqueeze(1) + self.log_p  # shape (n_j, n_i)
        term1 = -torch.logsumexp(log_terms, dim=0).sum() / self.n_i  # average over samples
        
        # Second term: +theta * (1/n_j) * sum_j w_j * log(w_j / w_j^0)
        if self.theta > 0:
            log_w0 = torch.log(self.w0 + eps)
            term2 = self.theta * torch.sum(w * (log_w - log_w0)) / self.n_j  # average over weights
        else:
            term2 = torch.tensor(0.0, dtype=torch.float64, device=self.device)
        
        return term1 + term2
        
    
    def optimize(self, lr=0.1, max_iter=10000, tol=1e-9, verbose=False):
        """
        Optimize weights using PyTorch Adam optimizers
        
        Args:
            lr: learning rate
            max_iter: maximum iterations
            tol: convergence tolerance
            verbose: print progress
        """
        # Use unconstrained parameterization: w = softmax(z)
        # Initialize with random normalized weights
        z_init = torch.randn(self.n_j, dtype=torch.float64, device=self.device)
        z = z_init.clone().detach().requires_grad_(True)
        
        # initialize Adam optimizer
        optimizer = torch.optim.Adam([z], lr=lr)
            
        losses = []
        for iteration in range(max_iter):
             optimizer.zero_grad()
                
             # Convert unconstrained z to normalized weights
             w = torch.softmax(z, dim=0)
                
             loss = self.compute_loss(w)
             loss.backward()
             optimizer.step()
                
             losses.append(loss.item())
                
             if verbose and iteration % 100 == 0:
                  print(f"Iter {iteration}: Loss = {loss.item():.6f}")
                
             # Check convergence
             if iteration > 10 and abs(losses[-1] - losses[-2]) < tol:
                 if verbose:
                     print(f"Converged at iteration {iteration}")
                 break
                    
        # Final weights
        with torch.no_grad():
            w_opt = torch.softmax(z, dim=0)
        
        return w_opt.cpu().numpy(), losses

Now we can finally infer the population of the 20 original models using the usual BioEM/CryoLike formula. We expect only the first and last models to be populated at 75%-25% as designed!

In [ ]:
# Initialize optimizer
opt = WeightOptimizer(log_probs_matrix, device=device)

# Optimize with Adam and find weights that maximize the BioEM posterior
w_opt, _ = opt.optimize()

In [ ]:
def plot_weights_comparison(x_opt, w_actual):
    """
    Plot comparison between optimized and actual weights with RMSE.
    
    Parameters:
        x_opt: numpy.array or torch.Tensor shape (n,) - optimized weights
        w_actual: numpy.array or torch.Tensor shape (n,) - actual/true weights
        save_path: optional path to save the figure
    
    Returns:
        rmse: float - Root Mean Squared Error between x_opt and w_actual
    """
    # Convert to numpy if needed
    if torch.is_tensor(x_opt):
        x_opt = x_opt.detach().cpu().numpy()
    if torch.is_tensor(w_actual):
        w_actual = w_actual.detach().cpu().numpy()
    
    # Calculate RMSE
    rmse = np.sqrt(np.mean((x_opt - w_actual)**2))
    
    # Create grouped bar plot
    fig, ax = plt.subplots(figsize=(10, 4))
    
    n_mod = len(x_opt)
    model_indexes = np.arange(1, n_mod + 1)
    bar_width = 0.35
    x_pos = np.arange(n_mod)
    
    # Plot both bars
    bars1 = ax.bar(x_pos - bar_width/2, x_opt, bar_width, 
                   label='CryoSB-Like weights', color='#3498db', 
                   alpha=0.8, edgecolor='black', linewidth=1.2)
    bars2 = ax.bar(x_pos + bar_width/2, w_actual, bar_width, 
                   label='True weights', color='#2ecc71', 
                   alpha=0.8, edgecolor='black', linewidth=1.2)
    
    # Labels and styling
    ax.set_xlabel('Model Index', fontsize=12, weight='bold')
    ax.set_ylabel('Weight', fontsize=12, weight='bold')
    ax.set_title(f'Weights Comparison - RMSE = {rmse:.6f}', 
                 fontsize=14, weight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(model_indexes)
    ax.set_xlim(-0.5, n_mod - 0.5)
    ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    
    plt.show()
    
    return rmse

Now we plot the inferred weights along with the ground truth (and the RMSE)

In [ ]:
rmse = plot_weights_comparison(w_opt, w_actual)

### Population inference (BME style)

In [ ]:
def run_theta_analysis(log_probs_matrix, x_true, w0=None, theta_max=1000.0, n_points=50, 
                       verbose=False, device='cpu'):
    """
    Run optimization over a grid of theta values and compute RMSE.
    
    Parameters:
        log_probs_matrix: torch.Tensor shape (n, m) - log likelihoods
        x0_prior: torch.Tensor or numpy.array shape (n,) - prior weights (optional, default: uniform)
        theta_max: maximum theta value
        n_points: number of theta values to test
        verbose: whether to print progress for each optimization
    
    Returns:
        theta_values: array of theta values
        weights_dict: dictionary mapping theta -> optimized weights
        rmse_values: array of RMSE values
    """
    
    # Create theta grid (linearly-spaced)
    theta_values = np.linspace(0.0, theta_max, n_points, endpoint=True)
    
    weights_dict = {}
    rmse_values = []
    losses_values = []
    
    print(f"Running optimization for {len(theta_values)} theta values...")
    
    for i, theta in enumerate(theta_values):
        if (i % 5 == 0):
            print(f"Progress: {i}/{len(theta_values)} - theta = {theta:.4f}")

        # Initialize optimizer
        opt = WeightOptimizer(log_probs_matrix, w0=w0, theta=theta, device=device)
        x_opt,l = opt.optimize()
        
        # Store weights in dictionary
        weights_dict[theta] = x_opt
        # and final loss
        losses_values.append(l[-1])
        
        # Compute RMSE with respect to prior weights
        rmse = np.sqrt(np.mean((x_opt - x_true)**2))
        rmse_values.append(rmse)
    
    rmse_values = np.array(rmse_values)
    losses_values = np.array(losses_values)
    
    print("Done!")
    return theta_values, weights_dict, rmse_values, losses_values

Now we infer the population of the 20 original models using the BioEM/CryoLike formula with an added regularization term - BME style. We do this for a range of theta from 0 to theta_max. We use as prior weights a uniform distribution.

In [ ]:
# Perform optimization for a range of thetas
# we don't provide prior weights w0 = uniform weights
theta_values, weights_dict, rmse_values, losses_values = run_theta_analysis(
    log_probs_matrix,
    w_actual,
    w0=None,
    theta_max=100.0,
    n_points=50,
    device=device
)

In [ ]:
def plot_rmse_vs_theta(theta_values, rmse_values):
    """
    Plot RMSE as a function of theta.
    
    Parameters:
        theta_values: array of theta values
        rmse_values: array of RMSE values
    """
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    
    # Linear scale
    ax.plot(theta_values, rmse_values, 'b-', linewidth=2, marker='o', markersize=4)
    ax.set_xlabel('θ (Regularization Strength)', fontsize=12)
    ax.set_ylabel('RMSE', fontsize=12)
    ax.set_title('RMSE vs θ', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print some statistics
    min_idx = np.argmin(rmse_values)
    print(f"\nMinimum RMSE: {rmse_values[min_idx]:.6f} at θ = {theta_values[min_idx]:.4f}")
    print(f"RMSE at θ=0: {rmse_values[0]:.6f}")

In [ ]:
# Plot results
plot_rmse_vs_theta(theta_values, rmse_values)
# Visualize set of weights for a given theta value
# Let's start with theta=0.0
rmse = plot_weights_comparison(weights_dict[0.0], w_actual)
# And let's plot also theta=theta_max (should be uniform-ish, i.e. the prior)
rmse = plot_weights_comparison(weights_dict[100.0], w_actual)